In [11]:
import torch
import torch.nn as nn
import sys
import numpy as np
from pathlib import Path


sys.path.append(str(Path.cwd().parent))
from src.metrics import evaluate
from src.data import load_ratings, time_split, build_id_maps

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

ratings = load_ratings()
train, test_warm = time_split(ratings)
user_to_idx, movie_to_idx = build_id_maps(train)
n_users, n_movies = len(user_to_idx), len(movie_to_idx)

print(f"n_users: {n_users}, n_movies: {n_movies}")

n_users: 5400, n_movies: 3662


In [ ]:
# precompute seen movies per user (as dense indices)
train_u = train["user_id"].map(user_to_idx).values
train_m = train["movie_id"].map(movie_to_idx).values

seen_by_user = {}
for u, m in zip(train_u, train_m):
    seen_by_user.setdefault(u, set()).add(m)


def sample_negative(user_idx: int) -> int:
    """Draw a random movie index this user has NOT interacted with."""
    seen = seen_by_user[user_idx]
    while True:
        random_movie = np.random.randint(n_movies)
        if random_movie not in seen:
            return random_movie

In [ ]:
def make_batch(batch_indices):
    """Given row indices into the training data, build a batch of
    positives and sampled negatives.

    Returns (users, movies, labels) as tensors.
    """
    users = train_u[batch_indices]      # positive users (dense idx)
    pos_movies = train_m[batch_indices] # the movies they actually interacted with

    # sample one negative movie per user in the batch
    neg_movies = np.array([sample_negative(u) for u in users])

    # stack positives then negatives
    batch_users = np.concatenate([users, users])           # same users twice
    batch_movies = np.concatenate([pos_movies, neg_movies])
    batch_labels = np.concatenate([np.ones(len(users)), np.zeros(len(users))])

    # convert to tensors
    batch_users = torch.tensor(batch_users, dtype=torch.long)
    batch_movies = torch.tensor(batch_movies, dtype=torch.long)
    batch_labels = torch.tensor(batch_labels, dtype=torch.float)

    return batch_users, batch_movies, batch_labels